In [3]:
import sys
from pathlib import Path

print("当前 Notebook 内核：", sys.executable)

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import InMemorySaver
from langgraph_py.deepagent import load_model

# Notebook 中没有 __file__；这里使用当前工作目录。
workspace = Path.cwd()
workspace.mkdir(exist_ok=True)

model = load_model()  # 复用本项目 .env 中的模型配置
backend = FilesystemBackend(root_dir=workspace, virtual_mode=True)
deep_agent = create_deep_agent(model=model, backend=backend, checkpointer=InMemorySaver())
print("文件工作区：", workspace)


当前 Notebook 内核： /Users/yang/fe/fe/.venv/bin/python
文件工作区： /Users/yang/fe/fe/langgraph-py/src/langgraph_py


In [5]:
config = {"configurable": {"thread_id": "file-write-demo"}}
result = deep_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "把'你好，Deep Agent！'写入 /hello.txt。"
    }]},
    config=config,
)

print(result)

{'messages': [HumanMessage(content="把'你好，Deep Agent！'写入 /hello.txt。", additional_kwargs={}, response_metadata={}, id='98e13eb0-1b52-487b-9a06-02b8aeaa7c17'), AIMessage(content='我现在来创建那个文件。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 2750, 'total_tokens': 2816, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 2688, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-1-flash-261017', 'system_fingerprint': None, 'id': '021792744471082593075d5bfc3e0cbf107ff469a4d1c91cd547a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a167e5-b85f-7a42-afb4-3081dee65e0f-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/he

In [ ]:
from langchain_core.messages import ToolMessage

config = {"configurable": {"thread_id": "file-write-demo"}}
read_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": "请调用 read_file 读取 /hello.txt，原样告诉我文件内容。"}]},
    config=config,  # 沿用写入时的 thread_id
)
reads = [m for m in read_result["messages"] if isinstance(m, ToolMessage) and m.name == "read_file"]
print(reads[-1].content if reads else "模型没有调用 read_file：" + str(read_result["messages"][-1].content))